In [ ]:
# [CELL 1] CORE DEPENDENCIES & REPRODUCIBILITY
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import joblib

# Scikit-Learn & ML Modules
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

# Guarantee absolute reproducibility across runs
os.environ['PYTHONHASHSEED'] = '42'
np.random.seed(42)

print("All foundational machine learning dependencies successfully loaded.")

In [ ]:
# [CELL 2] GLOBAL EXPERIMENT CONFIGURATION
CONFIG = {
    # Dataset Configuration
"FILE_PATH": "/BTC/OHLCV_Dataset.csv",

    # Model Selection Architecture
    # Options: 'SVR', 'XGBOOST'
    "MODEL_TYPE": "XGBOOST",

    # Time-Series Windowing Parameters
    "LOOK_BACK": 60,                        # Input window
    "TARGET_STEPS": 2,                      # Multi-step Horizon

    # Data Splitting Hyperparameters
    "TRAIN_SPLIT": 0.80,                    # 80% Training, 20% Testing (Chronological split)

    # Feature Engineering Parameters
    "FEATURES": ["Open", "High", "Low", "Close", "Volume"],
    "TARGET_FEATURE": "Close",              # The specific metric to forecast

    # Thesis Artifacts Directory
    "OUTPUT_DIR": "./thesis_results"
}

# Ensure directory exists for saving publication graphs & CSV metrics
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

# Apply a clean, standard styling context for high-resolution publication-ready figures
plt.style.use('default')

print(f"Cell 2 executed successfully: Configuration initialized for {CONFIG['MODEL_TYPE']} model.")

In [ ]:
# [CELL 3] EXPLORATORY DATA ANALYSIS (EDA) ENGINE
def run_publication_eda(df, config):
    """
    Generates high-resolution, publication-ready statistical visualizations
    and saves core metrics into the designated output directory.
    """
    print("## Running Dataset Structural Diagnostics...")
    print(f"Shape of Dataset: {df.shape}")
    print("\nData Types & Info:")
    df.info()

    # 1. Statistical Summary (Saved as CSV for Thesis Tables)
    summary = df[config["FEATURES"]].describe()
    summary_path = os.path.join(config["OUTPUT_DIR"], "statistical_summary.csv")
    summary.to_csv(summary_path)
    print(f"\n Statistical summary saved to: {summary_path}")

    # 2. Correlation Matrix Heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[config["FEATURES"]].corr(), annot=True, cmap="coolwarm", fmt=".4f", square=True)
    plt.title("Feature Correlation Matrix", fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_correlation_matrix.png"), dpi=300)
    plt.show()

    # 3. Macro Trends: Closing Price & Volume Over Time
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Target Feature (Close Price)
    axes[0].plot(df.index, df[config["TARGET_FEATURE"]], color='#1f77b4', linewidth=1)
    axes[0].set_title("Bitcoin Historical Closing Price Trend", fontweight='bold', fontsize=12)
    axes[0].set_ylabel("Price (USDT)")
    axes[0].grid(True, linestyle='--', alpha=0.5)

    # Trading Volume
    axes[1].fill_between(df.index, df["Volume"], color='#ff7f0e', alpha=0.5)
    axes[1].set_title("Historical Trading Volume", fontweight='bold', fontsize=12)
    axes[1].set_ylabel("Volume")
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_macro_trends.png"), dpi=300)
    plt.show()

    # 4. Feature Distributions Histograms
    df[config["FEATURES"]].hist(bins=50, figsize=(14, 9), color='darkblue', grid=True, edgecolor='black', alpha=0.7)
    plt.suptitle("Feature Distributions Analysis", fontweight='bold', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_feature_distributions.png"), dpi=300)
    plt.show()

print("Cell 3 executed successfully: Publication-ready EDA engine defined.")

In [ ]:
# [CELL 4] DATA PREPARATION & MULTI-STEP SEQUENCE ENGINE
def load_and_index_dataset(config):
    """
    Loads the financial time-series data and establishes a clean
    DatetimeIndex to handle rigorous chronological ordering.
    """
    df = pd.read_csv(config["FILE_PATH"])
    date_col = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    if date_col:
        df[date_col[0]] = pd.to_datetime(df[date_col[0]], format='mixed')
        df.set_index(date_col[0], inplace=True)
    return df

def generate_sequences(data, look_back, target_idx, target_steps):
    """
    Generates input-output windows.
    y captures a continuous vector of 'target_steps' ahead.
    """
    X, y = [], []
    for i in range(len(data) - look_back - target_steps + 1):
        X.append(data[i : (i + look_back), :])
        y.append(data[(i + look_back) : (i + look_back + target_steps), target_idx])

    return np.array(X), np.array(y)

def prep_thesis_data(df, config):
    """
    Transforms raw dataframe into normalized training and testing tensors
    while strictly avoiding forward-looking/data leakage bias.
    """
    data_matrix = df[config["FEATURES"]].values
    target_idx = config["FEATURES"].index(config["TARGET_FEATURE"])

    # Chronological training/testing split (No random shuffling)
    split_boundary = int(len(data_matrix) * config["TRAIN_SPLIT"])
    train_data = data_matrix[:split_boundary]
    test_data = data_matrix[split_boundary:]

    # Fit scaler ONLY on training data
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_train = scaler.fit_transform(train_data)
    scaled_test = scaler.transform(test_data)

    X_train, y_train = generate_sequences(scaled_train, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])
    X_test, y_test = generate_sequences(scaled_test, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])

    return X_train, y_train, X_test, y_test, scaler, target_idx

print("Cell 4 executed successfully: Multi-step data preparation pipeline fully optimized.")

In [ ]:
# [CELL 5] UNIFIED ML MODEL FACTORY
def build_ml_architecture(model_type):
    """
    Academic Factory Pattern generating Machine Learning baselines.
    Leverages MultiOutputRegressor to strictly match the Deep Learning
    multi-step vector outputs.
    """
    model_type = model_type.upper()


    # 1. SUPPORT VECTOR REGRESSION (SVR)

    if model_type == "SVR":
        # Using research-grade defaults; C and gamma can be optimized later
        base_model = SVR(kernel='rbf', C=10.0, gamma='scale', epsilon=0.01)
        # Wrap in MultiOutputRegressor to handle target_steps > 1
        model = MultiOutputRegressor(base_model)


    # 2. EXTREME GRADIENT BOOSTING (XGBOOST)

    elif model_type == "XGBOOST":
        base_model = XGBRegressor(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='reg:squarederror',
            random_state=42
        )
        # XGBoost handles multi-output natively, but MultiOutputRegressor
        # guarantees interface consistency with SVR and Sklearn standards.
        model = MultiOutputRegressor(base_model)

    else:
        raise ValueError(f" Model Type '{model_type}' is missing from the Cell 5 Factory configuration.")

    return model

print("Master Factory updated. ML baselines (SVR, XGBoost) fully active and uniform!")

In [ ]:
# [CELL 6] VALIDATION & EVALUATION METRICS ENGINE
def execute_walk_forward_validation(df, config, folds=3):
    """
    Executes Walk-Forward Time-Series Cross-Validation.
    Handles automatic flattening of the 3D tensor to 2D for ML algorithms.
    """
    print("## Running Walk-Forward Time-Series Cross Validation...")
    data_matrix = df[config["FEATURES"]].values
    target_idx = config["FEATURES"].index(config["TARGET_FEATURE"])
    fold_size = len(data_matrix) // (folds + 1)
    fold_scores = []

    for f in range(folds):
        train_end = fold_size * (f + 1)
        test_end = train_end + fold_size

        train_block = data_matrix[:train_end]
        test_block = data_matrix[train_end:test_end]

        fold_scaler = MinMaxScaler()
        scaled_train = fold_scaler.fit_transform(train_block)
        scaled_test = fold_scaler.transform(test_block)

        X_tr, y_tr = generate_sequences(scaled_train, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])
        X_te, y_te = generate_sequences(scaled_test, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])

        # FLATTEN 3D (Samples, Lookback, Features) to 2D (Samples, Lookback * Features) for ML models
        X_tr_flat = X_tr.reshape(X_tr.shape[0], -1)
        X_te_flat = X_te.reshape(X_te.shape[0], -1)

        fold_model = build_ml_architecture(config["MODEL_TYPE"])
        fold_model.fit(X_tr_flat, y_tr)

        preds = fold_model.predict(X_te_flat)
        fold_mse = mean_squared_error(y_te, preds)
        fold_scores.append(fold_mse)
        print(f"Fold {f+1}/{folds} Normalized Multi-step MSE: {fold_mse:.6f}")

    return np.mean(fold_scores)

print("Cell 6 executed successfully: Walk-forward validation adapted for Machine Learning.")

In [ ]:
# [CELL 7] LOAD DATA & RUN EDA
try:
    raw_df = load_and_index_dataset(CONFIG)
    print(" Dataset loaded successfully from specified path.")
except FileNotFoundError:
    print(f"Dataset not found at {CONFIG['FILE_PATH']}. Generating synthetic dummy data for pipeline testing.")
    dummy_dates = pd.date_range(start="2021-01-01", periods=5000, freq="5min")
    raw_df = pd.DataFrame(np.random.randn(5000, 5), columns=CONFIG["FEATURES"], index=dummy_dates)
    raw_df["Close"] = 50000 + raw_df["Close"].cumsum() * 10

run_publication_eda(raw_df, CONFIG)
print(" Cell 7 executed successfully: Data pipeline ingestion and EDA diagnostics completed.")

In [ ]:
# [CELL 8] CROSS-VALIDATION EXECUTION & MAIN TENSOR TRAIN-TEST SPLIT
print("==============================================================")
print("     PHASE 1: EXECUTING TIME-SERIES CROSS-VALIDATION          ")
print("==============================================================")

mean_cv_loss = execute_walk_forward_validation(raw_df, CONFIG, folds=3)
print(f"\n Overall Mean Cross-Validation MSE: {mean_cv_loss:.6f}")

print("\n==============================================================")
print("     PHASE 2: PREPARING PRODUCTION TRAIN-TEST TENSORS        ")
print("==============================================================")

X_train, y_train, X_test, y_test, scaler, target_idx = prep_thesis_data(raw_df, CONFIG)

print("\n Verification of Multi-Step Tensor Structural Dimensions:")
print(f"• X_train shape (Samples, Lookback, Features): {X_train.shape}")
print(f"• y_train shape (Samples, Target Steps):       {y_train.shape}")
print(f"• X_test shape  (Samples, Lookback, Features): {X_test.shape}")
print(f"• y_test shape  (Samples, Target Steps):       {y_test.shape}")

print("\n Cell 8 executed successfully: Dataset split validated.")

In [ ]:
# [CELL 9] CENTRALIZED MODEL TRAINING ENGINE
print("==============================================================")
print(f"     PHASE 3: TRAINING {CONFIG['MODEL_TYPE']} ARCHITECTURE")
print("==============================================================")

model = build_ml_architecture(CONFIG["MODEL_TYPE"])

# FLATTEN 3D tensors to 2D for Scikit-Learn / XGBoost compatibility
X_train_flat = X_train.reshape(X_train.shape[0], -1)

print(f"\n Initiating training sequence...")
start_train_time = time.time()

# Fit the ML model (Non-iterative/Internal iteration compared to deep learning epochs)
model.fit(X_train_flat, y_train)

training_duration = time.time() - start_train_time
print(f"\n Training cycle finalized in: {training_duration:.2f} seconds.")

print("\n==============================================================")
print("     PHASE 3.5: SAVING MODEL ARTIFACTS                       ")
print("==============================================================")

# Save the trained ML model structurally similarly to Keras models
model_save_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['MODEL_TYPE']}_master_model.pkl")
joblib.dump(model, model_save_path)
print(f" Model securely saved to {model_save_path}")

# (Note: Learning curve plots are omitted here as ML models like SVR do not
# inherently track epoch-by-epoch loss histories during standard .fit() calls.)

print("\n Cell 9 executed successfully: Model trained and artifacts exported.")

In [ ]:
# [CELL 10] FULL 13-ELEMENT EVALUATION & THESIS REPORTING ENGINE
print("==============================================================")
print("     PHASE 4: COMPLETE SYSTEM EVALUATION & PERFORMANCE        ")
print("==============================================================")

# 1. Track Inference Efficiency Latency
print(f" Generating multi-step predictions using trained {CONFIG['MODEL_TYPE']} model...")
# Flatten evaluation tensor
X_test_flat = X_test.reshape(X_test.shape[0], -1)

start_inf_time = time.time()
scaled_preds = model.predict(X_test_flat)
inference_duration = time.time() - start_inf_time

# 2. Compute Structural Complexity Metrics (Adjusted for ML)
# ML models don't have standard backprop 'parameters', utilizing model artifact file size
total_model_params = 0  # Not directly comparable to neural networks
try:
    estimated_size_mb = os.path.getsize(os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['MODEL_TYPE']}_master_model.pkl")) / (1024 * 1024)
except Exception:
    estimated_size_mb = 0.0

recorded_train_time = globals().get('training_duration', 0.0)

# 3. Perform Mathematical De-scaling to obtain actual prices (USDT)
num_features = len(CONFIG["FEATURES"])

print("\n Reversing min-max normalization to obtain actual Bitcoin prices (USDT)...")
def inverse_transform_multistep(scaled_matrix, scaler, target_idx, num_features):
    samples, steps = scaled_matrix.shape
    flat_scaled = scaled_matrix.reshape(-1, 1)
    dummy_matrix = np.zeros((flat_scaled.shape[0], num_features))
    dummy_matrix[:, target_idx] = flat_scaled[:, 0]
    inv_flat = scaler.inverse_transform(dummy_matrix)[:, target_idx]
    return inv_flat.reshape(samples, steps)

actual_prices = inverse_transform_multistep(y_test, scaler, target_idx, num_features)
predicted_prices = inverse_transform_multistep(scaled_preds, scaler, target_idx, num_features)

# Flatten arrays for point-by-point statistical validation metrics
actual_flat = actual_prices.flatten()
predicted_flat = predicted_prices.flatten()

# 4. Direct In-Cell Computation of the 9 Accuracy Metrics
print("\n Computing validation errors...")
mse = mean_squared_error(actual_flat, predicted_flat)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actual_flat, predicted_flat)
median_ae = median_absolute_error(actual_flat, predicted_flat)
max_error = np.max(np.abs(actual_flat - predicted_flat))

# Safe calculation for percentage errors
mape = np.mean(np.abs((actual_flat - predicted_flat) / np.where(actual_flat == 0, 1e-5, actual_flat))) * 100
smape = 200 * np.mean(np.abs(predicted_flat - actual_flat) / (np.abs(actual_flat) + np.abs(predicted_flat) + 1e-5))
r2 = r2_score(actual_flat, predicted_flat)

# Directional Accuracy computed per forecast step
last_known_scaled = X_test[:, -1, target_idx]
dummy = np.zeros((len(last_known_scaled), num_features))
dummy[:, target_idx] = last_known_scaled
last_known_actual = scaler.inverse_transform(dummy)[:, target_idx]

step_das = []
for step in range(CONFIG["TARGET_STEPS"]):
    true_dir = np.sign(actual_prices[:, step] - last_known_actual)
    pred_dir = np.sign(predicted_prices[:, step] - last_known_actual)
    step_da = np.mean(true_dir == pred_dir) * 100
    step_das.append(step_da)
    print(f"   Step {step+1} Directional Accuracy: {step_da:.2f}%")

directional_accuracy = np.mean(step_das)

# 5. Compile All 13 Thesis Elements into a Single Master Dictionary
master_thesis_report = {
    # --- SECTION A: ACCURACY METRICS ---
    "MSE": mse,
    "RMSE": rmse,
    "MAE": mae,
    "Median_AE": median_ae,
    "Max_Error": max_error,
    "MAPE(%)": mape,
    "SMAPE(%)": smape,
    "R2_Score": r2,
    "Directional_Accuracy(%)": directional_accuracy,

    # --- SECTION B: COMPUTATIONAL EFFICIENCY METRICS ---
    "Training_Time(s)": recorded_train_time,
    "Inference_Time(s)": inference_duration,
    "Total_Model_Parameters": total_model_params,  # 0 for ML algorithms
    "Estimated_Model_Size(MB)": estimated_size_mb
}

# 6. Display Clean Summary Table
print("\n" + "="*55)
print(f"   MASTER THESIS BENCHMARK REPORT: {CONFIG['MODEL_TYPE']} ENGINE")
print("="*55)
print(f" {'METRIC ELEMENT':<30} | {'VALUE / SCORE':<20}")
print("-" * 55)
for metric_key, value in master_thesis_report.items():
    if "%" in metric_key:
        print(f" • {metric_key:<28} | {value:.4f}%")
    elif "Time" in metric_key:
        print(f" • {metric_key:<28} | {value:.2f} seconds")
    elif "Parameters" in metric_key:
        if value == 0:
            print(f" • {metric_key:<28} | N/A (ML Model)")
        else:
            print(f" • {metric_key:<28} | {int(value):,}")
    elif "Size" in metric_key:
        print(f" • {metric_key:<28} | {value:.4f} MB")
    else:
        print(f" • {metric_key:<28} | {value:.6f}")
print("======================================================")

# 7. Export Unified Results to CSV Spreadsheet
report_df = pd.DataFrame(list(master_thesis_report.items()), columns=["Thesis Metric Element", "Value Score"])
csv_save_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['MODEL_TYPE']}_complete_thesis_metrics.csv")
report_df.to_csv(csv_save_path, index=False)
print(f" Comprehensive metrics sheet successfully saved to: {csv_save_path}")

# 8. Plot Forecast Comparison Chart
plt.figure(figsize=(14, 6), dpi=300)
plot_tail = 150
plt.plot(actual_flat[-plot_tail:], label="Actual BTC Price (USDT)", color="#1f77b4", linewidth=2)
plt.plot(predicted_flat[-plot_tail:], label=f"Predicted BTC Price ({CONFIG['MODEL_TYPE']})",
         color="#d62728", linestyle="--", linewidth=1.8)

plt.title(f"Bitcoin Price Tracking Analysis ({CONFIG['MODEL_TYPE']} Multi-Step Framework)",
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Continuous Multi-Step Timeline Slices (1-Hour Steps)", fontsize=11, labelpad=8)
plt.ylabel("Price (USDT)", fontsize=11, labelpad=8)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", fontsize=10, frameon=True)
plt.tight_layout()

chart_save_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['MODEL_TYPE']}_prediction_plot.png")
plt.savefig(chart_save_path, dpi=300)
plt.show()

print("\n Cell 10 executed successfully: All 13 metrics are captured, logged, and plotted.")